# Task 2 — Full CRUD with `requests` (POST, PUT, PATCH, DELETE)

**Topic:** Complete REST cycle — Create, Read, Update, Delete — using Python's `requests` library

**Kya seekhenge (what we'll learn):**
- `requests.post()` se naya resource banana
- `requests.put()` se poora resource replace karna
- `requests.patch()` se sirf kuch fields update karna
- `requests.delete()` se resource remove karna
- `headers` aur `json=` parameter ka sahi use
- `raise_for_status()` se proper error handling
- Reusable functions banana jo har CRUD operation ko wrap karein

**API used:** `https://jsonplaceholder.typicode.com` — ye fake API hai, matlab POST/PUT/PATCH/DELETE
"success" response dete hain lekin data server pe actually save NAHI hota (isliye har baar fresh data milega).
Practice ke liye ye perfect hai kyunke koi cheez permanently kharab nahi hoti.

> Reminder: Task 1 mein humne sirf GET (Read) kiya tha. Ye notebook baaki 4 operations cover karta hai.


In [1]:
import requests

BASE_URL = "https://jsonplaceholder.typicode.com"
print("Ready! Base URL:", BASE_URL)


Ready! Base URL: https://jsonplaceholder.typicode.com


## 1. CREATE — `requests.post()`

Naya resource banane ke liye POST use hota hai. Hum `json=` parameter mein
Python dictionary bhejte hain, `requests` khud usko JSON string mein convert kar deta hai
aur `Content-Type: application/json` header bhi khud set kar deta hai.

**Success status code: 201 (Created)**


In [2]:
new_post = {
    "title": "Mera Pehla API Post",
    "body": "Ye post requests library se POST method use kar ke bana hai.",
    "userId": 1
}

response = requests.post(f"{BASE_URL}/posts", json=new_post)

print("Status code:", response.status_code)   # 201 expect kar rahe hain
print("Server ne kya wapis bheja:")
print(response.json())   # server naya 'id' assign kar ke wapis bhejta hai (fake, save nahi hota)


Status code: 201
Server ne kya wapis bheja:
{'title': 'Mera Pehla API Post', 'body': 'Ye post requests library se POST method use kar ke bana hai.', 'userId': 1, 'id': 101}


## 2. READ (Recap) — `requests.get()`

CRUD complete karne ke liye Read bhi dobara dikha rahe hain — jo humne Task 1 mein seekha tha.


In [3]:
response = requests.get(f"{BASE_URL}/posts/1")
print("Status:", response.status_code)
print(response.json())


Status: 200
{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


## 3. UPDATE (Full) — `requests.put()`

PUT poora resource replace kar deta hai — matlab agar koi field bhejna bhool gaye,
to wo field response mein missing ho sakti hai (real APIs mein). Isliye PUT mein
**saari fields bhejni chahiye**, sirf jo update karni hain wo nahi.

**Success status code: 200 (OK)**


In [4]:
updated_post = {
    "id": 1,
    "title": "Updated Title — Poora Post Replace Kiya",
    "body": "PUT method poora resource replace karta hai, isliye saari fields di hain.",
    "userId": 1
}

response = requests.put(f"{BASE_URL}/posts/1", json=updated_post)

print("Status code:", response.status_code)   # 200 expect
print(response.json())


Status code: 200
{'id': 1, 'title': 'Updated Title — Poora Post Replace Kiya', 'body': 'PUT method poora resource replace karta hai, isliye saari fields di hain.', 'userId': 1}


## 4. UPDATE (Partial) — `requests.patch()`

PATCH sirf wo fields update karta hai jo aap bhejte hain — baaki data waisa hi rehta hai.
Real-world mein PATCH zyada use hota hai kyunke poora object bhejna zaroori nahi hota.

**Success status code: 200 (OK)**


In [5]:
# Sirf title change kar rahe hain — body aur userId ko touch nahi kiya
partial_update = {
    "title": "Sirf Title Change Hui — Baaki Same Hai"
}

response = requests.patch(f"{BASE_URL}/posts/1", json=partial_update)

print("Status code:", response.status_code)
print(response.json())


Status code: 200
{'userId': 1, 'id': 1, 'title': 'Sirf Title Change Hui — Baaki Same Hai', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


## 5. DELETE — `requests.delete()`

Resource ko remove karne ke liye. Zyada tar APIs body ki zaroorat nahi hoti DELETE mein,
bas URL mein id kaafi hai.

**Success status code: 200 (OK) ya 204 (No Content)**


In [6]:
response = requests.delete(f"{BASE_URL}/posts/1")

print("Status code:", response.status_code)
print("Response body (aksar empty hota hai delete ke baad):", response.json())


Status code: 200
Response body (aksar empty hota hai delete ke baad): {}


## 6. Proper Error Handling — `raise_for_status()`

Har baar `if response.status_code == 200` likhna thakane wala hai.
`response.raise_for_status()` khud check karta hai — agar error status ho (4xx ya 5xx)
to ye ek Exception raise kar deta hai jo hum `try/except` mein pakad sakte hain.


In [7]:
def safe_request(method, url, **kwargs):
    """
    Ye helper function koi bhi request bhejta hai aur errors ko sahi tarah handle karta hai.
    method: 'get', 'post', 'put', 'patch', 'delete' (string)
    url: poora endpoint
    kwargs: extra cheezein jese json=, params= waghera
    """
    try:
        response = requests.request(method, url, **kwargs)
        response.raise_for_status()   # agar 4xx/5xx hai to yahan Exception aa jayegi
        return response.json()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error aaya: {e}")
        return None
    except requests.exceptions.ConnectionError:
        print("Connection nahi ban saka — internet check karo.")
        return None
    except requests.exceptions.Timeout:
        print("Request time out ho gayi.")
        return None


# Test 1: valid request
result = safe_request("get", f"{BASE_URL}/posts/1")
print("Valid request result:", result)
print()

# Test 2: invalid request (jaan bujh kar galat URL)
result = safe_request("get", f"{BASE_URL}/posts/999999999")
print("Invalid request result:", result)


Valid request result: {'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}

HTTP Error aaya: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/posts/999999999
Invalid request result: None


---
## Exercises — Ab Aapki Bari

Har `# TODO` ki jagah apna code likho.


### Exercise 1 — Naya Comment POST Karo
`https://jsonplaceholder.typicode.com/comments` pe POST request bhejo, body mein `postId`, `name`, `email`, `body` fields ke saath. Status code aur naya `id` print karo.

In [ ]:
# TODO: comment dictionary banao (postId, name, email, body fields ke saath)
# TODO: POST request bhejo /comments endpoint pe
# TODO: status code aur response ka id print karo



### Exercise 2 — Todo Ko PATCH Karo
Todo id `5` ka `completed` field `True` kar do PATCH request se (`https://jsonplaceholder.typicode.com/todos/5`).

In [ ]:
# TODO: PATCH request bhejo, sirf {"completed": True} bhejo
# TODO: response print karo



### Exercise 3 — Ek User Delete Karo
User id `10` ko DELETE karo (`https://jsonplaceholder.typicode.com/users/10`) aur status code check karo.

In [ ]:
# TODO: DELETE request bhejo
# TODO: status code print karo (200 ya 204 expect karo)



### Exercise 4 — Mini Project: Simple Todo Manager Functions

Neeche 3 functions likho jo mil kar ek chhota "Todo Manager" banate hain:

1. `create_todo(title, user_id)` — POST se naya todo banaye, return kare naya todo dictionary
2. `mark_complete(todo_id)` — PATCH se `completed: True` set kare, return kare updated todo
3. `delete_todo(todo_id)` — DELETE kare, return kare True/False (kaamyabi ke hisaab se)

Sab functions mein `safe_request()` helper (upar wala) use karo — dobara likhne ki zaroorat nahi.

Aakhir mein teeno functions ko test karo.


In [ ]:
# TODO: create_todo function

# TODO: mark_complete function

# TODO: delete_todo function


# TODO: teeno ko test karo


---
## Wrap-up

| Method  | Kaam                        | Success Code |
|---------|------------------------------|--------------|
| POST    | Naya resource banana          | 201          |
| GET     | Data read karna                | 200          |
| PUT     | Poora resource replace karna   | 200          |
| PATCH   | Sirf kuch fields update karna  | 200          |
| DELETE  | Resource remove karna          | 200 / 204    |

**Publish karne se pehle:** README mein likhen ke kaunsi API use hui aur kya seekha.
